**Verify NVIDIA GPU Availability**

In [ ]:
!nvidia-smi

**Split images into train and validation folders**

In [ ]:
!python ../src/train_val_split.py --targetpath="../split-data" --datapath="../data" --train_pct=0.9

**Configure Training**

In [ ]:
# Python function to automatically create data.yaml config file
# 1. Reads "classes.txt" file to get list of class names
# 2. Creates data dictionary with correct paths to folders, number of classes, and names of classes
# 3. Writes data in YAML format to data.yaml

import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  # Read class.txt to get class names
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  # Create data dictionary
  data = {
      'path': '../data',
      'train': '../split-data/train/images',
      'val': '../split-data/validation/images',
      'nc': number_of_classes,
      'names': classes
  }

  # Write data to YAML file
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Define path to classes.txt and run function
path_to_classes_txt = '../data/classes.txt'
path_to_data_yaml = '../data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nFile contents:\n')
!cat ../data.yaml

**Training**

In [ ]:
!yolo detect train data=../data.yaml model=yolo11s.pt epochs=10 imgsz=640

**Test model**

In [ ]:
!yolo detect predict model=runs/detect/train-7/weights/best.pt source=../split-data/validation/images save=True

In [ ]:
import glob
from IPython.display import Image, display
for image_path in glob.glob(f'runs/detect/predict/*.jpg')[:10]:
  display(Image(filename=image_path, height=400))
  print('\n')

In [ ]:
!python ../src/yolo_detect.py --model runs/detect/train-7/weights/best.pt --source ../video.mp4 --resolution 1280x720